#  DINOv2 + YOLO Bird Classification Pipeline


## Step 1: Setup and Installations

In [ ]:
# Install required packages
!pip install ultralytics transformers torch torchvision scikit-learn pandas numpy pillow opencv-python tqdm -q

print("All packages installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 31.4 MB/s eta 0:00:00
✓ All packages installed!


In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

print("Google Drive mounted!")

Mounted at /content/drive
✓ Google Drive mounted!


In [ ]:
# Imports
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from transformers import AutoImageProcessor, Dinov2Model
from ultralytics import YOLO
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image
from tqdm import tqdm
import json
from datetime import datetime
import cv2
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

print("All imports successful!")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✓ All imports successful!


## Step 2: Configuration

In [4]:
# Paths
BASE_PATH = Path("/content/drive/MyDrive/bird_classification/bdma-07-competition-2026/BDMA7_project_files")
TRAIN_IMG_DIR = BASE_PATH / "train_images"
VAL_IMG_DIR = BASE_PATH / "val_images"
TEST_IMG_DIR = BASE_PATH / "test_images/mistery_cat"
OUTPUT_DIR = BASE_PATH / "dinov2_results"
OUTPUT_DIR.mkdir(exist_ok=True)

CROPPED_DIR = OUTPUT_DIR / "cropped_images"
CROPPED_DIR.mkdir(exist_ok=True)

# Training configuration
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
EPOCHS = 20
N_FOLDS = 3
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("="*80)
print("🦅 DINOv2 + YOLO Bird Classification Pipeline")
print("="*80)
print(f"Device: {DEVICE}")
print(f"Data directory exists: {BASE_PATH.exists()}")
print(f"Output directory: {OUTPUT_DIR}")

🦅 DINOv2 + YOLO Bird Classification Pipeline
Device: cuda
Data directory exists: True
Output directory: /content/drive/MyDrive/bird_classification/bdma-07-competition-2026/BDMA7_project_files/dinov2_results


## Step 3: YOLO Bird Detection & Cropping

This step detects birds in images and crops them with a margin. This focuses the model on the bird rather than background.

In [ ]:
def crop_bird_from_image(image_path, output_path, detector, margin=0.2):
    """
    Detect bird in image and crop to bounding box with margin

    Args:
        image_path: Path to input image
        output_path: Path to save cropped image
        detector: YOLO detector
        margin: Margin to add around bounding box (0.2 = 20%)

    Returns:
        bool: True if successful, False otherwise
    """
    try:
        # Read image
        img = cv2.imread(str(image_path))
        if img is None:
            return False

        h, w = img.shape[:2]

        # Detect birds (class 14 in COCO dataset)
        results = detector(image_path, classes=[14], verbose=False)

        if len(results[0].boxes) == 0:
            # No bird detected - save original with resize
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            pil_img = Image.fromarray(img_rgb)
            pil_img = pil_img.resize((224, 224))
            pil_img.save(output_path)
            return True

        # Get largest detection (assume it's the main bird)
        boxes = results[0].boxes.xyxy.cpu().numpy()
        areas = [(box[2] - box[0]) * (box[3] - box[1]) for box in boxes]
        largest_idx = np.argmax(areas)
        box = boxes[largest_idx]

        x1, y1, x2, y2 = box

        # Add margin
        box_w = x2 - x1
        box_h = y2 - y1

        x1 = max(0, x1 - box_w * margin)
        y1 = max(0, y1 - box_h * margin)
        x2 = min(w, x2 + box_w * margin)
        y2 = min(h, y2 + box_h * margin)

        # Crop
        cropped = img[int(y1):int(y2), int(x1):int(x2)]

        # Convert to RGB and save
        cropped_rgb = cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB)
        pil_img = Image.fromarray(cropped_rgb)
        pil_img.save(output_path)

        return True

    except Exception as e:
        print(f"Error processing {image_path}: {e}")
        return False

print("Cropping function defined")

✓ Cropping function defined


In [ ]:
# Load YOLO detector
print("Loading YOLO detector...")
detector = YOLO('yolov8x.pt')  # Using largest model for best accuracy
print("YOLO loaded")

Loading YOLO detector...
✓ YOLO loaded


In [7]:
# Process training images
print("\n" + "="*80)
print("Processing Training Images")
print("="*80)

train_cropped_dir = CROPPED_DIR / "train"
train_cropped_dir.mkdir(exist_ok=True)

# Iterate through class folders
class_folders = [d for d in TRAIN_IMG_DIR.iterdir() if d.is_dir()]
print(f"Found {len(class_folders)} class folders in training set")

total_train_images = 0
for class_folder in tqdm(class_folders, desc="Processing classes"):
    class_name = class_folder.name

    # Create corresponding output directory
    output_class_dir = train_cropped_dir / class_name
    output_class_dir.mkdir(exist_ok=True)

    # Get all images in this class
    class_images = list(class_folder.glob('*.jpg')) + list(class_folder.glob('*.png'))
    total_train_images += len(class_images)

    # Crop each image
    for img_path in class_images:
        output_path = output_class_dir / img_path.name
        if not output_path.exists():
            crop_bird_from_image(img_path, output_path, detector)

print(f"\n✓ {total_train_images} training images cropped and saved to {train_cropped_dir}")


Processing Training Images
Found 20 class folders in training set


Processing classes: 100%|██████████| 20/20 [14:46<00:00, 44.32s/it]


✓ 1082 training images cropped and saved to /content/drive/MyDrive/bird_classification/bdma-07-competition-2026/BDMA7_project_files/dinov2_results/cropped_images/train


In [8]:
# Process validation images
print("\n" + "="*80)
print("Processing Validation Images")
print("="*80)

val_cropped_dir = CROPPED_DIR / "val"
val_cropped_dir.mkdir(exist_ok=True)

# Iterate through class folders
class_folders = [d for d in VAL_IMG_DIR.iterdir() if d.is_dir()]
print(f"Found {len(class_folders)} class folders in validation set")

total_val_images = 0
for class_folder in tqdm(class_folders, desc="Processing classes"):
    class_name = class_folder.name

    # Create corresponding output directory
    output_class_dir = val_cropped_dir / class_name
    output_class_dir.mkdir(exist_ok=True)

    # Get all images in this class
    class_images = list(class_folder.glob('*.jpg')) + list(class_folder.glob('*.png'))
    total_val_images += len(class_images)

    # Crop each image
    for img_path in class_images:
        output_path = output_class_dir / img_path.name
        if not output_path.exists():
            crop_bird_from_image(img_path, output_path, detector)

print(f"\n✓ {total_val_images} validation images cropped and saved to {val_cropped_dir}")


Processing Validation Images
Found 20 class folders in validation set


Processing classes: 100%|██████████| 20/20 [01:27<00:00,  4.37s/it]


✓ 103 validation images cropped and saved to /content/drive/MyDrive/bird_classification/bdma-07-competition-2026/BDMA7_project_files/dinov2_results/cropped_images/val


In [9]:
# Process test images
print("\n" + "="*80)
print("Processing Test Images")
print("="*80)

test_cropped_dir = CROPPED_DIR / "test"
test_cropped_dir.mkdir(exist_ok=True)

test_images = list(TEST_IMG_DIR.glob('*.jpg')) + list(TEST_IMG_DIR.glob('*.png'))
print(f"Found {len(test_images)} test images")

for img_path in tqdm(test_images, desc="Cropping test images"):
    output_path = test_cropped_dir / img_path.name
    if not output_path.exists():
        crop_bird_from_image(img_path, output_path, detector)

print(f"✓ Test images cropped and saved to {test_cropped_dir}")


Processing Test Images
Found 400 test images


Cropping test images: 100%|██████████| 400/400 [00:01<00:00, 265.60it/s]

✓ Test images cropped and saved to /content/drive/MyDrive/bird_classification/bdma-07-competition-2026/BDMA7_project_files/dinov2_results/cropped_images/test


## Step 4: Load Data from Directory Structure

In [10]:
# Build dataset from directory structure
# Structure: train_images/species_name/*.jpg
#           val_images/species_name/*.jpg

def load_images_from_directory(base_dir, cropped_dir):
    """
    Load images from directory structure where each class is a folder

    Args:
        base_dir: Original images directory
        cropped_dir: Cropped images directory (mirrors structure)

    Returns:
        DataFrame with columns: original_path, cropped_path, class_name
    """
    data = []

    # Iterate through class folders
    class_folders = [d for d in base_dir.iterdir() if d.is_dir()]

    for class_folder in class_folders:
        class_name = class_folder.name

        # Get all images in this class folder
        image_files = list(class_folder.glob('*.jpg')) + list(class_folder.glob('*.png'))

        for img_path in image_files:
            # Cropped image path mirrors the structure
            relative_path = img_path.relative_to(base_dir)
            cropped_path = cropped_dir / relative_path

            data.append({
                'original_path': str(img_path),
                'cropped_path': str(cropped_path),
                'class_name': class_name,
                'image_name': img_path.name
            })

    return pd.DataFrame(data)

print("Loading data from directory structure...")

# Load train and val data
train_df = load_images_from_directory(TRAIN_IMG_DIR, train_cropped_dir)
val_df = load_images_from_directory(VAL_IMG_DIR, val_cropped_dir)

print(f"Train images: {len(train_df)}")
print(f"Val images: {len(val_df)}")

# Combine into single dataset
combined_df = pd.concat([train_df, val_df], ignore_index=True)

# Create class mapping (alphabetical for consistency)
unique_classes = sorted(combined_df['class_name'].unique())
class_mapping = {name: idx for idx, name in enumerate(unique_classes)}
combined_df['label'] = combined_df['class_name'].map(class_mapping).astype('int64')

# Use cropped_path as full_path for training
combined_df['full_path'] = combined_df['cropped_path']

print(f"\nTotal samples: {len(combined_df)}")
print(f"Number of classes: {len(class_mapping)}")
print(f"Samples per class: ~{len(combined_df) / len(class_mapping):.0f}")

# Show class distribution
print("\nClass distribution:")
class_counts = combined_df['class_name'].value_counts().sort_index()
for class_name, count in class_counts.items():
    print(f"  {class_mapping[class_name]:2d}. {class_name:30s}: {count:3d} images")

# Save class mapping
with open(OUTPUT_DIR / 'class_mapping.json', 'w') as f:
    json.dump(class_mapping, f, indent=2)

print(f"\n✓ Class mapping saved to {OUTPUT_DIR / 'class_mapping.json'}")

Loading data from directory structure...
Train images: 1082
Val images: 103

Total samples: 1185
Number of classes: 20
Samples per class: ~59

Class distribution:
   0. American_Crow                 :  60 images
   1. Black_billed_Cuckoo           :  60 images
   2. Bobolink                      :  60 images
   3. Brandt_Cormorant              :  59 images
   4. Brewer_Blackbird              :  59 images
   5. Bronzed_Cowbird               :  60 images
   6. Brown_Creeper                 :  59 images
   7. Eastern_Towhee                :  60 images
   8. Fish_Crow                     :  60 images
   9. Gray_Catbird                  :  59 images
  10. Gray_crowned_Rosy_Finch       :  59 images
  11. Groove_billed_Ani             :  60 images
  12. Indigo_Bunting                :  60 images
  13. Lazuli_Bunting                :  58 images
  14. Painted_Bunting               :  58 images
  15. Red_winged_Blackbird          :  60 images
  16. Rusty_Blackbird               :  60 images
  17

## Step 5: Setup DINOv2 Model

In [11]:
# Load DINOv2 processor and model
print("Loading facebook/dinov2-base...")
processor = AutoImageProcessor.from_pretrained('facebook/dinov2-base')
dinov2_backbone = Dinov2Model.from_pretrained('facebook/dinov2-base')

# Freeze the backbone
for param in dinov2_backbone.parameters():
    param.requires_grad = False

dinov2_backbone = dinov2_backbone.to(DEVICE)

print("✓ DINOv2 loaded and frozen")
print(f"  Embedding dimension: 768")
print(f"  Parameters frozen: {sum(p.numel() for p in dinov2_backbone.parameters()):,}")

Loading facebook/dinov2-base...


preprocessor_config.json:   0%|          | 0.00/436 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


config.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

✓ DINOv2 loaded and frozen
  Embedding dimension: 768
  Parameters frozen: 86,580,480


In [12]:
# Define transforms
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=processor.image_mean, std=processor.image_std)
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=processor.image_mean, std=processor.image_std)
])

print("✓ Transforms defined")

✓ Transforms defined


In [13]:
# Dataset class
class BirdDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row['full_path']).convert('RGB')
        label = row['label']

        if self.transform:
            image = self.transform(image)

        return image, label

print("✓ Dataset class defined")

✓ Dataset class defined


In [14]:
# DINOv2 Classifier Model
class DINOv2Classifier(nn.Module):
    """
    DINOv2 with frozen backbone + trainable classifier head
    """
    def __init__(self, backbone, num_classes, hidden_dim=512, dropout=0.3):
        super().__init__()
        self.backbone = backbone

        # Classification head with more capacity for fine-grained features
        self.classifier = nn.Sequential(
            nn.LayerNorm(768),
            nn.Dropout(dropout),
            nn.Linear(768, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(hidden_dim, 256),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        # Extract features from frozen DINOv2
        with torch.no_grad():
            outputs = self.backbone(x)
            # Use CLS token (first token)
            features = outputs.last_hidden_state[:, 0]

        # Classify
        logits = self.classifier(features)
        return logits

print("✓ DINOv2Classifier defined")

✓ DINOv2Classifier defined


## Step 6: Training Function

In [ ]:
def train_dinov2_classifier(train_loader, val_loader, fold_idx):
    """Train DINOv2 classifier with frozen backbone"""

    print(f"\n{'='*60}")
    print(f"TRAINING FOLD {fold_idx + 1}")
    print(f"{'='*60}")

    # Create model
    model = DINOv2Classifier(
        backbone=dinov2_backbone,
        num_classes=len(class_mapping),
        hidden_dim=512,
        dropout=0.3
    )
    model = model.to(DEVICE)

    # Count parameters
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,} ({100*trainable_params/total_params:.2f}%)")

    # Loss and optimizer
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = torch.optim.AdamW(
        model.classifier.parameters(),
        lr=LEARNING_RATE,
        weight_decay=1e-4
    )

    # Learning rate scheduler
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    best_val_acc = 0.0
    best_model_state = None
    patience = 5
    patience_counter = 0

    for epoch in range(EPOCHS):
        # Training
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0

        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS} [Train]', leave=False)
        for images, labels in pbar:
            images, labels = images.to(DEVICE), labels.to(DEVICE)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = outputs.max(1)
            train_total += labels.size(0)
            train_correct += predicted.eq(labels).sum().item()

            pbar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'acc': f'{100.*train_correct/train_total:.2f}%'
            })

        train_acc = 100. * train_correct / train_total
        avg_train_loss = train_loss / len(train_loader)

        # Validation
        model.eval()
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for images, labels in tqdm(val_loader, desc=f'Epoch {epoch+1}/{EPOCHS} [Val]', leave=False):
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                outputs = model(images)
                _, predicted = outputs.max(1)
                val_total += labels.size(0)
                val_correct += predicted.eq(labels).sum().item()

        val_acc = 100. * val_correct / val_total

        print(f"Epoch {epoch+1:2d}: Train Loss: {avg_train_loss:.4f} | "
              f"Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")

        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = model.state_dict().copy()
            patience_counter = 0
            print(f" New best validation accuracy!")
        else:
            patience_counter += 1

        # Early stopping
        if patience_counter >= patience:
            print(f"  Early stopping triggered after {epoch+1} epochs")
            break

        scheduler.step()

    print(f"\nFold {fold_idx + 1} Best Val Acc: {best_val_acc:.2f}%")

    return {
        'model_state': best_model_state,
        'best_val_acc': best_val_acc
    }

print("✓ Training function defined")

✓ Training function defined


## Step 7: K-Fold Training

In [16]:
print("="*80)
print("🏋️ STARTING K-FOLD CROSS-VALIDATION")
print("="*80)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
fold_results = []

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(combined_df, combined_df['label'])):
    # Create datasets
    train_fold_df = combined_df.iloc[train_idx]
    val_fold_df = combined_df.iloc[val_idx]

    train_dataset = BirdDataset(train_fold_df, transform=train_transform)
    val_dataset = BirdDataset(val_fold_df, transform=val_transform)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    print(f"\nFold {fold_idx + 1}: Train={len(train_dataset)}, Val={len(val_dataset)}")

    # Train model
    result = train_dinov2_classifier(train_loader, val_loader, fold_idx)
    fold_results.append(result)

print("\n✓ All folds trained!")

🏋️ STARTING K-FOLD CROSS-VALIDATION

Fold 1: Train=790, Val=395

TRAINING FOLD 1
Total parameters: 87,112,212
Trainable parameters: 531,732 (0.61%)


Epoch  1: Train Loss: 1.5497 | Train Acc: 73.42% | Val Acc: 91.90%
  ✨ New best validation accuracy!


Epoch  2: Train Loss: 0.8349 | Train Acc: 92.41% | Val Acc: 94.68%
  ✨ New best validation accuracy!


Epoch  3: Train Loss: 0.7629 | Train Acc: 94.05% | Val Acc: 94.43%


Epoch  4: Train Loss: 0.7260 | Train Acc: 96.20% | Val Acc: 94.18%


Epoch  5: Train Loss: 0.7197 | Train Acc: 95.70% | Val Acc: 93.92%


... (23 outputs left out)

Epoch  8: Train Loss: 0.6678 | Train Acc: 98.48% | Val Acc: 94.68%


Epoch  9: Train Loss: 0.6664 | Train Acc: 98.35% | Val Acc: 93.92%


Epoch 10: Train Loss: 0.6584 | Train Acc: 98.61% | Val Acc: 93.42%


Epoch 11: Train Loss: 0.6492 | Train Acc: 99.24% | Val Acc: 94.43%
  Early stopping triggered after 11 epochs

Fold 3 Best Val Acc: 95.19%

✓ All folds trained!


## Step 8: Save Models

In [17]:
# Save models
model_save_dir = OUTPUT_DIR / 'dinov2_models'
model_save_dir.mkdir(exist_ok=True)

for fold_idx, result in enumerate(fold_results):
    save_path = model_save_dir / f'fold_{fold_idx}.pth'
    torch.save(result['model_state'], save_path)
    print(f"Saved fold {fold_idx} to {save_path}")

print(f"\n✓ All models saved to {model_save_dir}")

Saved fold 0 to /content/drive/MyDrive/bird_classification/bdma-07-competition-2026/BDMA7_project_files/dinov2_results/dinov2_models/fold_0.pth
Saved fold 1 to /content/drive/MyDrive/bird_classification/bdma-07-competition-2026/BDMA7_project_files/dinov2_results/dinov2_models/fold_1.pth
Saved fold 2 to /content/drive/MyDrive/bird_classification/bdma-07-competition-2026/BDMA7_project_files/dinov2_results/dinov2_models/fold_2.pth

✓ All models saved to /content/drive/MyDrive/bird_classification/bdma-07-competition-2026/BDMA7_project_files/dinov2_results/dinov2_models


## Step 9: Training Summary

In [18]:
print("="*80)
print("📊 TRAINING SUMMARY")
print("="*80)

val_accs = [r['best_val_acc'] for r in fold_results]
mean_val_acc = np.mean(val_accs)
std_val_acc = np.std(val_accs)

print(f"\nValidation Accuracies:")
for fold_idx, acc in enumerate(val_accs):
    print(f"  Fold {fold_idx + 1}: {acc:.2f}%")

print(f"\nMean: {mean_val_acc:.2f}% ± {std_val_acc:.2f}%")

# Save metadata
metadata = {
    'timestamp': datetime.now().isoformat(),
    'model': 'dinov2-base + classifier',
    'strategy': 'yolo_crop + frozen_backbone',
    'epochs': EPOCHS,
    'learning_rate': LEARNING_RATE,
    'batch_size': BATCH_SIZE,
    'mean_val_acc': mean_val_acc,
    'std_val_acc': std_val_acc,
    'fold_accs': val_accs,
    'class_mapping': class_mapping
}

with open(OUTPUT_DIR / 'training_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"\n✅ Training complete!")
print(f"📁 Models saved to {model_save_dir}")
print(f"📊 Expected test accuracy: {mean_val_acc:.1f}% ± {std_val_acc:.1f}%")

📊 TRAINING SUMMARY

Validation Accuracies:
  Fold 1: 95.19%
  Fold 2: 93.16%
  Fold 3: 95.19%

Mean: 94.51% ± 0.95%

✅ Training complete!
📁 Models saved to /content/drive/MyDrive/bird_classification/bdma-07-competition-2026/BDMA7_project_files/dinov2_results/dinov2_models
📊 Expected test accuracy: 94.5% ± 1.0%


## Step 10: Evaluation on Train/Val Set

In [19]:
print("="*80)
print("📊 EVALUATING ON TRAIN/VAL SET")
print("="*80)

eval_dataset = BirdDataset(combined_df, transform=val_transform)

all_fold_preds = []
all_fold_labels = []

for fold in range(N_FOLDS):
    print(f"\nFold {fold + 1}/{N_FOLDS}")

    # Load model
    model = DINOv2Classifier(
        backbone=dinov2_backbone,
        num_classes=len(class_mapping),
        hidden_dim=512,
        dropout=0.3
    )

    model_path = model_save_dir / f'fold_{fold}.pth'
    model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    model = model.to(DEVICE)
    model.eval()

    eval_loader = DataLoader(
        eval_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    fold_preds = []
    fold_labels = []

    with torch.no_grad():
        for images, labels in tqdm(eval_loader, desc=f'Fold {fold+1} Eval'):
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(images)
            predictions = torch.argmax(outputs, dim=1)

            fold_preds.extend(predictions.cpu().numpy())
            fold_labels.extend(labels.cpu().numpy())

    fold_preds = np.array(fold_preds)
    fold_labels = np.array(fold_labels)

    all_fold_preds.append(fold_preds)
    all_fold_labels.append(fold_labels)

    accuracy = (fold_preds == fold_labels).mean()
    print(f"Fold {fold + 1} Accuracy: {accuracy:.4f}")

# Ensemble
all_fold_preds = np.array(all_fold_preds)
all_labels = all_fold_labels[0]

ensemble_preds, _ = stats.mode(all_fold_preds, axis=0, keepdims=False)

print(f"\n{'='*80}")
print("ENSEMBLE RESULTS")
print(f"{'='*80}")
print(f"Ensemble Accuracy: {(ensemble_preds == all_labels).mean():.4f}")

📊 EVALUATING ON TRAIN/VAL SET

Fold 1/3


Fold 1 Eval: 100%|██████████| 38/38 [00:19<00:00,  1.98it/s]


Fold 1 Accuracy: 0.9764

Fold 2/3


Fold 2 Eval: 100%|██████████| 38/38 [00:19<00:00,  2.00it/s]


Fold 2 Accuracy: 0.9722

Fold 3/3


Fold 3 Eval: 100%|██████████| 38/38 [00:18<00:00,  2.04it/s]

Fold 3 Accuracy: 0.9747

ENSEMBLE RESULTS
Ensemble Accuracy: 0.9890


## Step 11: Generate Predictions for Mystery Test Set

In [20]:
print("="*80)
print("🔮 GENERATING PREDICTIONS FOR MYSTERY TEST SET")
print("="*80)

# Create test dataset
class TestDataset(Dataset):
    def __init__(self, image_paths, transform=None):
        self.image_paths = image_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image, str(img_path.name)

# Load cropped test images
test_images = sorted(list(test_cropped_dir.glob('*.jpg')))
print(f"Found {len(test_images)} test images")

test_dataset = TestDataset(test_images, transform=val_transform)

# Generate predictions from all folds
all_fold_predictions = []

for fold in range(N_FOLDS):
    print(f"\nFold {fold + 1}/{N_FOLDS}")

    model = DINOv2Classifier(
        backbone=dinov2_backbone,
        num_classes=len(class_mapping),
        hidden_dim=512,
        dropout=0.3
    )

    model_path = model_save_dir / f'fold_{fold}.pth'
    model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    model = model.to(DEVICE)
    model.eval()

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    fold_preds = []
    filenames = []

    with torch.no_grad():
        for images, names in tqdm(test_loader, desc=f'Fold {fold+1} Predictions'):
            images = images.to(DEVICE)

            outputs = model(images)
            predictions = torch.argmax(outputs, dim=1)

            fold_preds.extend(predictions.cpu().numpy())
            filenames.extend(names)

    all_fold_predictions.append(np.array(fold_preds))

# Ensemble predictions
all_fold_predictions = np.array(all_fold_predictions)
ensemble_predictions, _ = stats.mode(all_fold_predictions, axis=0, keepdims=False)

print(f"\nGenerated {len(ensemble_predictions)} predictions")

🔮 GENERATING PREDICTIONS FOR MYSTERY TEST SET
Found 400 test images

Fold 1/3


Fold 1 Predictions: 100%|██████████| 13/13 [00:55<00:00,  4.28s/it]



Fold 2/3


Fold 2 Predictions: 100%|██████████| 13/13 [00:06<00:00,  2.07it/s]



Fold 3/3


Fold 3 Predictions: 100%|██████████| 13/13 [00:06<00:00,  2.03it/s]


Generated 400 predictions


## Step 12: Create Submission File

In [21]:
# Create submission dataframe
submission_df = pd.DataFrame({
    'image_name': filenames,
    'predicted_class_idx': ensemble_predictions
})

# Sort by image name
submission_df = submission_df.sort_values('image_name').reset_index(drop=True)

# Add class names for reference
idx_to_class = {idx: name for name, idx in class_mapping.items()}
submission_df['predicted_class_name'] = submission_df['predicted_class_idx'].map(idx_to_class)

print("Submission preview:")
print(submission_df.head(10))

# Save submission
submission_path = OUTPUT_DIR / 'test_predictions.csv'
submission_df[['image_name', 'predicted_class_idx']].to_csv(submission_path, index=False)

print(f"\n✅ Submission saved to: {submission_path}")

# Also save full version with names
submission_full_path = OUTPUT_DIR / 'test_predictions_with_names.csv'
submission_df.to_csv(submission_full_path, index=False)
print(f"✅ Full predictions saved to: {submission_full_path}")

Submission preview:
                                 image_name  predicted_class_idx  \
0  000c02a0-3687-4c0c-bdfc-394b56134ac6.jpg                    0   
1  0036f660-7d6e-46d4-822d-dbb71b20c788.jpg                    5   
2  0082f891-33f6-4057-aa63-c2f462ea66a5.jpg                   13   
3  01e21934-d737-4992-b712-cf544666bd54.jpg                    0   
4  0321cf4c-2a19-42b5-992c-f1d6d80af4c7.jpg                    9   
5  03b3a4cf-dccc-4874-bf30-809e5498a815.jpg                   18   
6  04ad8917-07dc-451a-a59c-f4fc8827e0ab.jpg                    4   
7  054c4510-d3a2-4b20-976e-2cf93f78ec81.jpg                    8   
8  05d54ffe-c3ef-41c8-94eb-44bdd2c77311.jpg                   13   
9  061c3569-07a0-4611-ad18-ffeec09fb828.jpg                   10   

      predicted_class_name  
0            American_Crow  
1          Bronzed_Cowbird  
2           Lazuli_Bunting  
3            American_Crow  
4             Gray_Catbird  
5     Yellow_breasted_Chat  
6         Brewer_Blackbird  

## Step 13: Prediction Summary

In [22]:
print("="*80)
print("PREDICTION SUMMARY")
print("="*80)
print(f"Total test images: {len(ensemble_predictions)}")
print(f"Number of classes predicted: {len(np.unique(ensemble_predictions))} out of {len(class_mapping)}")

print("\nPrediction distribution:")
for class_idx in range(len(class_mapping)):
    count = (ensemble_predictions == class_idx).sum()
    class_name = idx_to_class[class_idx]
    if count > 0:
        print(f"  {class_idx:2d} ({class_name:30s}): {count:3d} ({100*count/len(ensemble_predictions):.1f}%)")

print("\n" + "="*80)
print("✅ DONE! Predictions ready for submission.")
print("="*80)

PREDICTION SUMMARY
Total test images: 400
Number of classes predicted: 20 out of 20

Prediction distribution:
   0 (American_Crow                 ):  18 (4.5%)
   1 (Black_billed_Cuckoo           ):  21 (5.2%)
   2 (Bobolink                      ):  24 (6.0%)
   3 (Brandt_Cormorant              ):  21 (5.2%)
   4 (Brewer_Blackbird              ):  17 (4.2%)
   5 (Bronzed_Cowbird               ):  21 (5.2%)
   6 (Brown_Creeper                 ):  21 (5.2%)
   7 (Eastern_Towhee                ):  19 (4.8%)
   8 (Fish_Crow                     ):  26 (6.5%)
   9 (Gray_Catbird                  ):  20 (5.0%)
  10 (Gray_crowned_Rosy_Finch       ):  21 (5.2%)
  11 (Groove_billed_Ani             ):  19 (4.8%)
  12 (Indigo_Bunting                ):  15 (3.8%)
  13 (Lazuli_Bunting                ):  20 (5.0%)
  14 (Painted_Bunting               ):  20 (5.0%)
  15 (Red_winged_Blackbird          ):  14 (3.5%)
  16 (Rusty_Blackbird               ):  23 (5.8%)
  17 (Yellow_billed_Cuckoo          ):  

In [24]:
import pandas as pd

# ==============================
# Inputs (already available)
# ==============================
# ensemble_predictions : list or np.ndarray of ints (0–19, TRAINING ORDER)
# filenames            : list of image filenames (same order)
# OUTPUT_DIR           : pathlib.Path

# ==============================
# 1. TRAINING order mapping
#    (alphabetical, as used in training)
# ==============================
train_idx_to_class = {
    0: "American_Crow",
    1: "Black_billed_Cuckoo",
    2: "Bobolink",
    3: "Brandt_Cormorant",
    4: "Brewer_Blackbird",
    5: "Bronzed_Cowbird",
    6: "Brown_Creeper",
    7: "Eastern_Towhee",
    8: "Fish_Crow",
    9: "Gray_Catbird",
    10: "Gray_crowned_Rosy_Finch",
    11: "Groove_billed_Ani",
    12: "Indigo_Bunting",
    13: "Lazuli_Bunting",
    14: "Painted_Bunting",
    15: "Red_winged_Blackbird",
    16: "Rusty_Blackbird",
    17: "Yellow_billed_Cuckoo",
    18: "Yellow_breasted_Chat",
    19: "Yellow_headed_Blackbird",
}

# ==============================
# 2. CANONICAL mapping
#    (expected by evaluation)
# ==============================
canonical_class_mapping = {
    "Groove_billed_Ani": 0,
    "Red_winged_Blackbird": 1,
    "Rusty_Blackbird": 2,
    "Gray_Catbird": 3,
    "Brandt_Cormorant": 4,
    "Eastern_Towhee": 5,
    "Indigo_Bunting": 6,
    "Brewer_Blackbird": 7,
    "Painted_Bunting": 8,
    "Bobolink": 9,
    "Lazuli_Bunting": 10,
    "Yellow_headed_Blackbird": 11,
    "American_Crow": 12,
    "Fish_Crow": 13,
    "Brown_Creeper": 14,
    "Yellow_billed_Cuckoo": 15,
    "Yellow_breasted_Chat": 16,
    "Black_billed_Cuckoo": 17,
    "Gray_crowned_Rosy_Finch": 18,
    "Bronzed_Cowbird": 19,
}

# ==============================
# 3. Remap predictions
# ==============================
predicted_class_names = [
    train_idx_to_class[int(idx)] for idx in ensemble_predictions
]

predicted_class_idx = [
    canonical_class_mapping[name] for name in predicted_class_names
]

# ==============================
# 4. Build submission dataframe
# ==============================
submission_df = pd.DataFrame({
    "path": filenames,
    "class_idx": predicted_class_idx,
    "class_name": predicted_class_names,  # sanity only
})

submission_df = submission_df.sort_values("path").reset_index(drop=True)

print("Submission preview:")
print(submission_df.head(10))

# ==============================
# 5. Save submission files
# ==============================
submission_path = OUTPUT_DIR / "test_predictions.csv"
submission_df[["path", "class_idx"]].to_csv(
    submission_path, index=False
)
print(f"\n✅ Submission saved to: {submission_path}")

submission_full_path = OUTPUT_DIR / "test_predictions_with_names.csv"
submission_df.to_csv(submission_full_path, index=False)
print(f"✅ Full predictions saved to: {submission_full_path}")


Submission preview:
                                       path  class_idx  \
0  000c02a0-3687-4c0c-bdfc-394b56134ac6.jpg         12   
1  0036f660-7d6e-46d4-822d-dbb71b20c788.jpg         19   
2  0082f891-33f6-4057-aa63-c2f462ea66a5.jpg         10   
3  01e21934-d737-4992-b712-cf544666bd54.jpg         12   
4  0321cf4c-2a19-42b5-992c-f1d6d80af4c7.jpg          3   
5  03b3a4cf-dccc-4874-bf30-809e5498a815.jpg         16   
6  04ad8917-07dc-451a-a59c-f4fc8827e0ab.jpg          7   
7  054c4510-d3a2-4b20-976e-2cf93f78ec81.jpg         13   
8  05d54ffe-c3ef-41c8-94eb-44bdd2c77311.jpg         10   
9  061c3569-07a0-4611-ad18-ffeec09fb828.jpg         18   

                class_name  
0            American_Crow  
1          Bronzed_Cowbird  
2           Lazuli_Bunting  
3            American_Crow  
4             Gray_Catbird  
5     Yellow_breasted_Chat  
6         Brewer_Blackbird  
7                Fish_Crow  
8           Lazuli_Bunting  
9  Gray_crowned_Rosy_Finch  

✅ Submission saved to

In [34]:
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ==============================
# Paths
# ==============================
PRED_PATH = OUTPUT_DIR / "test_predictions.csv"
TRUE_PATH = OUTPUT_DIR / "test_labels.csv"

# ==============================
# Load data
# ==============================
pred_df = pd.read_csv(PRED_PATH)
true_df = pd.read_csv(TRUE_PATH)

# Expected columns:
# pred_df : image_name, predicted_class_idx
# true_df : image_name, true_class_idx

# ==============================
# Align on image_name
# ==============================
df = pred_df.merge(
    true_df,
    left_on="path",
    right_on="test_image_name",
    how="inner",
    validate="one_to_one",
    suffixes=("_pred", "_true")
)

print(f"Compared samples: {len(df)}")

# ==============================
# Metrics
# ==============================
y_true = df["class_idx_true"]
y_pred = df["class_idx_pred"]

accuracy = accuracy_score(y_true, y_pred)
print(f"\n✅ Accuracy: {accuracy:.4f}")

print("\n📊 Classification report:")
print(classification_report(y_true, y_pred, digits=4))

# ==============================
# Confusion matrix
# ==============================
cm = confusion_matrix(y_true, y_pred)
cm_df = pd.DataFrame(cm)

print("\n🧩 Confusion matrix:")
print(cm_df)

# ==============================
# Inspect mismatches (optional)
# ==============================
errors = df[df["class_idx_pred"] != df["class_idx_true"]]

print(f"\n❌ Misclassified samples: {len(errors)}")

print("\nSample errors:")
print(errors.head(10))

# Save errors for inspection
errors_path = OUTPUT_DIR / "prediction_errors.csv"
errors.to_csv(errors_path, index=False)
print(f"\n📝 Error analysis saved to: {errors_path}")


Compared samples: 400

✅ Accuracy: 0.6400

📊 Classification report:
              precision    recall  f1-score   support

           0     0.6842    0.4194    0.5200        31
           1     0.6429    0.5625    0.6000        16
           2     0.2609    0.3000    0.2791        20
           3     0.7500    0.6522    0.6977        23
           4     0.8571    0.9000    0.8780        20
           5     0.8421    0.6957    0.7619        23
           6     0.8000    0.8000    0.8000        15
           7     0.1176    0.1429    0.1290        14
           8     0.6000    0.8000    0.6857        15
           9     0.5000    0.6667    0.5714        18
          10     0.7000    0.8235    0.7568        17
          11     0.6842    0.8125    0.7429        16
          12     0.5000    0.4737    0.4865        19
          13     0.2308    0.4615    0.3077        13
          14     0.7619    1.0000    0.8649        16
          15     0.8095    0.7083    0.7556        24
          16 